In [1]:
import pandas as pd
import numpy as np


In [2]:
raw = pd.read_excel('DA_BH.xlsx', sheet_name = 'Sales')
Sku_master = pd.read_excel('DA_BH.xlsx', sheet_name = 'SKU_Master')

raw.head()


,invoice_id,date,store_id,sku,qty,unit_price,discount_pct,promo_code,net_amount
0,HD260700000,2026-07-01,CT01,SKU045,1,613000.0,0.0,NaN,613000
1,HD260700001,2026-07-01,CT01,SKU038,1,329000.0,0.0,NaN,329000
2,HD260700002,2026-07-01,CT01,SKU009,2,261000.0,0.0,NaN,522000
3,HD260700003,2026-07-01,CT01,SKU002,3,322000.0,0.0,NaN,966000
4,HD260700004,2026-07-01,CT01,SKU022,2,630000.0,0.0,NaN,1260000


In [4]:
df = raw.copy()
log = {"rows_raw": len(raw)}
log['SKU sai'] = int((df['sku'] != df['sku'].str.strip().str.upper()).sum())
df['sku'] = df['sku'].str.strip().str.upper()

In [5]:
df.head()

,invoice_id,date,store_id,sku,qty,unit_price,discount_pct,promo_code,net_amount
0,HD260700000,2026-07-01,CT01,SKU045,1,613000.0,0.0,NaN,613000
1,HD260700001,2026-07-01,CT01,SKU038,1,329000.0,0.0,NaN,329000
2,HD260700002,2026-07-01,CT01,SKU009,2,261000.0,0.0,NaN,522000
3,HD260700003,2026-07-01,CT01,SKU002,3,322000.0,0.0,NaN,966000
4,HD260700004,2026-07-01,CT01,SKU022,2,630000.0,0.0,NaN,1260000


In [7]:
before = len(df)

df = df.drop_duplicates(subset = 'invoice_id', keep = 'first')
log['dòng bị trùng'] = before - len(df)


In [8]:
df.isna().sum()

invoice_id         0
date               0
store_id          11
sku                0
qty                0
unit_price        10
discount_pct       0
promo_code      3954
net_amount         0
dtype: int64

In [11]:
log['thieu_store_id'] = int(df['store_id'].isna().sum())
df['store_id'] = df['store_id'].fillna('UNKNOWN')


In [15]:
log['Thiếu init_price'] = int(df['unit_price'].isna().sum())

#lấy giá từ bảng SKU master
price = Sku_master.set_index('sku')['list_price']
df['unit_price'] = df['sku'].map(price)
df.head()
df.isna().sum()

invoice_id         0
date               0
store_id           0
sku                0
qty                0
unit_price         0
discount_pct       0
promo_code      3954
net_amount         0
dtype: int64

In [16]:
log['đồng trả hàng'] = int((df['qty'] < 0).sum())
log["rows_clean"] = len(df)
log["net_revenue_raw"] = raw["net_amount"].sum()
log["net_revenue_clean"] = df["net_amount"].sum()

df.to_csv("sales_clean.csv", index=False)
pd.Series(log).to_csv("cleaning_log.csv", header=["value"])
print(pd.Series(log))

rows_raw                   4122
SKU sai                      30
dòng bị trùng                 0
thieu_store_id               11
Thiếu init_price             10
đồng trả hàng                25
rows_clean                 4087
net_revenue_raw      3028513100
net_revenue_clean    3004457800
dtype: int64
